In [ ]:
# 01_raw_data_inspection.ipynb

# =========================
# IMPORTS
# =========================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")  # Seaborn styling

# =========================
# LOAD RAW DATA
# =========================
# Load the original results CSV
results = pd.read_csv("../data/raw/results.csv")

# Quick look at the first rows
results.head()

# =========================
# DATA INSPECTION
# =========================

# Check data types and non-null counts
results.info()

# Check for missing values
print("\nMissing values per column:")
print(results.isna().sum())

# Quick overview of tournaments
print("\nTop 10 tournaments by number of matches:")
print(results['tournament'].value_counts().head(10))

# Quick overview of total number of matches
print(f"\nTotal matches in raw dataset: {results.shape[0]}")

# =========================
# BASIC DESCRIPTIVE STATISTICS
# =========================

# Convert 'date' column to datetime
results['date'] = pd.to_datetime(results['date'], errors='coerce')

# Extract year for simple analysis
results['year'] = results['date'].dt.year

# Basic statistics for scores
print("\nHome score statistics:")
print(results['home_score'].describe())
print("\nAway score statistics:")
print(results['away_score'].describe())

# Total goals per match
results['total_goals'] = results['home_score'] + results['away_score']
print("\nTotal goals per match statistics:")
print(results['total_goals'].describe())

# Matches per decade (quick trend)
# Aggregate number of matches per decade (temporary, no new column in dataset)
matches_per_decade = (results.groupby(results['year'] // 10 * 10).size().reset_index(name='num_matches'))

# Rename decade column for clarity
matches_per_decade.rename(columns={'year': 'decade'}, inplace=True)

print("\nNumber of matches per decade:")
print(matches_per_decade)


# =========================
# FILTER AND SAVE CLEAN CSV
# =========================

# Filter matches: from 1990 onwards and exclude friendlies (for EDA)
eda_data = results[(results['year'] >= 1990)].copy()

# Filter matches: from 2000 onwards and exclude friendlies (for model training)
training_data = results[(results['year'] >= 2000) & (results['tournament'] != 'Friendly')].copy()

# Save processed CSV for model training
training_data.to_csv("../data/processed/results_2000_onwards.csv", index=False)
print("\nFiltered CSV saved to 'results_2000_onwards.csv'")

# =========================
# PLOTS FOR EDA
# =========================

# 1️⃣ Number of matches per year (trend)
# Aggregate number of matches per year
matches_per_year = eda_data.groupby('year').size().reset_index(name='num_matches')

# Plot
plt.figure(figsize=(14,6))
sns.lineplot(data=matches_per_year, x='year', y='num_matches', marker='o')
plt.title("Number of international matches per year (1990+)")
plt.xlabel("Year")
plt.ylabel("Number of matches")
plt.tight_layout()
plt.show()

# 2️⃣ Distribution of total goals per match
plt.figure(figsize=(12,6))
sns.histplot(eda_data['total_goals'], bins=range(0,11), kde=False, color="skyblue")
plt.title("Distribution of total goals per match (1990+)")
plt.xlabel("Total goals in match")
plt.ylabel("Number of matches")
plt.xticks(range(0,11))
plt.tight_layout()
plt.show()

# 3️⃣ Optional: Top 10 teams by total wins
# Calculate home/away wins
eda_data['home_win'] = eda_data['home_score'] > eda_data['away_score']
eda_data['away_win'] = eda_data['away_score'] > eda_data['home_score']

home_wins = eda_data.groupby('home_team')['home_win'].sum()
away_wins = eda_data.groupby('away_team')['away_win'].sum()
total_wins = home_wins.add(away_wins, fill_value=0).sort_values(ascending=False)
top_10 = total_wins.head(10)

plt.figure(figsize=(12,6))
sns.barplot(x=top_10.values, y=top_10.index, palette="viridis")
plt.title("Top 10 teams by total wins (1990+)")
plt.xlabel("Total wins")
plt.ylabel("Team")
plt.tight_layout()
plt.show()

# =========================
# END OF NOTEBOOK
# =========================
